# Bonus: Jinja2 Prompt Templates + PromptManager

---

## Why template your prompts?

Hard-coded prompts are fragile. Template-based prompts give you:

| Problem | Solution |
|---------|----------|
| Variables scattered across code | Single `.j2` file, one render call |
| No logic in prompts | Jinja2 `for`, `if`, filters |
| No metadata / versioning | YAML frontmatter (name, version, author) |
| Hard to inspect placeholders | `get_template_info()` lists them automatically |

### How it works

```
prompts/templates/my_prompt.j2
        │
        ├── YAML frontmatter  →  metadata (name, version, …)
        └── Jinja2 body       →  {{ variables }}, {% for %}, {% if %}
                                        │
                              PromptManager.get_prompt()
                                        │
                                  rendered string  →  LLM
```

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from langfuse_demo.prompt_manager import PromptManager

pm = PromptManager()
print(f"Templates dir: {pm.template_dir}")

Templates dir: /Users/marcospaulopaivapereira/Documents/Projetos/celfocus/langfuse/prompts/templates


---

## Part 1: `{% for %}` — Iterating over a list

**Use case**: prompt that receives a _list_ of ML experiments and asks the LLM to pick the best one.

Key Jinja2 concepts:
- `{% for item in items %}` ... `{% endfor %}` — loop over any iterable
- `{{ loop.index }}` — 1-based counter (use `loop.index0` for 0-based)
- `{{ item.field }}` — access dict keys or object attributes

In [2]:
# Read the raw template file so we can see the source
template_path = pm.template_dir / "for_example.j2"
print(template_path.read_text())

---
name: experiment-evaluator
description: Evaluates a list of ML experiments and recommends the best model
version: "1.0"
author: data-science-team
---
You are a machine learning expert. Analyze the following experiments and recommend the best model.

Question: {{ question }}

## Experiment Results:
{% for exp in experiments %}
{{ loop.index }}. **{{ exp.name }}**
   - Accuracy: {{ exp.accuracy }}
   - F1 Score: {{ exp.f1 }}
   - Training Time: {{ exp.training_time }}
{% endfor %}

Provide a clear recommendation with justification.



In [3]:
# Inspect template metadata + auto-detected placeholders
info = pm.get_template_info("for_example")
print("Metadata:")
for k, v in info.items():
    print(f"  {k}: {v}")

Metadata:
  name: experiment-evaluator
  description: Evaluates a list of ML experiments and recommends the best model
  version: 1.0
  author: data-science-team
  placeholders: {'experiments', 'question'}


In [4]:
# Render with real data
experiments = [
    {"name": "XGBoost baseline",  "accuracy": "0.842", "f1": "0.831", "training_time": "12s"},
    {"name": "LightGBM tuned",    "accuracy": "0.871", "f1": "0.865", "training_time": "9s"},
    {"name": "Random Forest v2",  "accuracy": "0.853", "f1": "0.847", "training_time": "34s"},
]

rendered = pm.get_prompt(
    "for_example",
    question="Which model should we deploy to production?",
    experiments=experiments,
)

print("=" * 60)
print(rendered)
print("=" * 60)

You are a machine learning expert. Analyze the following experiments and recommend the best model.

Question: Which model should we deploy to production?

## Experiment Results:
1. **XGBoost baseline**
   - Accuracy: 0.842
   - F1 Score: 0.831
   - Training Time: 12s
2. **LightGBM tuned**
   - Accuracy: 0.871
   - F1 Score: 0.865
   - Training Time: 9s
3. **Random Forest v2**
   - Accuracy: 0.853
   - F1 Score: 0.847
   - Training Time: 34s

Provide a clear recommendation with justification.


---

## Part 2: `{% if %}` — Conditional blocks

**Use case**: RAG-style Q&A prompt. When context is retrieved, include it. When not, fall back to general knowledge.

Key Jinja2 concepts:
- `{% if condition %}` ... `{% endif %}` — include block only if truthy
- `{% if value is not none %}` — explicit `None` check (Python `None` maps to Jinja2 `none`)
- Same template → different output depending on runtime data

In [5]:
template_path = pm.template_dir / "if_example.j2"
print(template_path.read_text())

---
name: rag-assistant
description: Q&A with optional RAG context — answers from context if available, general knowledge otherwise
version: "1.0"
author: data-science-team
---
You are a helpful data science assistant.
{% if context is not none %}

Use ONLY the context below to answer. If the answer isn't there, say so explicitly.

## Context:
{{ context }}
{% endif %}

## Question:
{{ question }}

Answer clearly and concisely.



In [6]:
# Render WITHOUT context (no RAG retrieval — general knowledge path)
rendered_no_ctx = pm.get_prompt(
    "if_example",
    question="What is the difference between precision and recall?",
    context=None,
)

print("--- WITHOUT context ---")
print(rendered_no_ctx)

--- WITHOUT context ---
You are a helpful data science assistant.

## Question:
What is the difference between precision and recall?

Answer clearly and concisely.


In [7]:
# Render WITH context (RAG retrieved a relevant doc)
retrieved_context = (
    "Precision = TP / (TP + FP). Measures how many predicted positives are truly positive. "
    "Recall = TP / (TP + FN). Measures how many actual positives were caught. "
    "F1 = harmonic mean of precision and recall. Used when class imbalance is present."
)

rendered_with_ctx = pm.get_prompt(
    "if_example",
    question="What is the difference between precision and recall?",
    context=retrieved_context,
)

print("--- WITH context ---")
print(rendered_with_ctx)

--- WITH context ---
You are a helpful data science assistant.

Use ONLY the context below to answer. If the answer isn't there, say so explicitly.

## Context:
Precision = TP / (TP + FP). Measures how many predicted positives are truly positive. Recall = TP / (TP + FN). Measures how many actual positives were caught. F1 = harmonic mean of precision and recall. Used when class imbalance is present.

## Question:
What is the difference between precision and recall?

Answer clearly and concisely.


---

## Summary

### `PromptManager` API

| Method | What it does |
|--------|--------------|
| `PromptManager(template_dir)` | Init — point to your `.j2` folder |
| `pm.get_prompt(name, **vars)` | Render template with variables → `str` |
| `pm.get_template_info(name)` | Returns frontmatter metadata + `placeholders` set |

### Jinja2 cheatsheet for prompts

```jinja2
{{ variable }}                      # insert value
{% for item in list %}...{% endfor %}  # iterate
{{ loop.index }}                    # 1-based counter inside for
{% if condition %}...{% endif %}    # conditional block
{% if value is not none %}          # None check
{{ text | upper }}                  # filters (upper, lower, truncate, …)
```

### File structure

```
prompts/templates/
├── for_example.j2   # {% for %} — list iteration
└── if_example.j2   # {% if %}  — optional context block
```

Each file has YAML frontmatter (name, version, author, description) + Jinja2 body.  